# RSNA Knee — Spec 01 on Kaggle

Runs the whole of Spec 01 (EDA + frozen folds) where the data is already mounted locally,
so no DICOM ever transits through Drive.

**Before running — two notebook settings (right-hand panel):**
1. *Input* → add the competition dataset.
2. *Internet* → **ON** (needed for `git clone` and `pip install`).

Run the cells top to bottom. Outputs land in `/kaggle/working/` and the last cell zips
them for download.

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/LIMAMMohamedlimam/rsna-knee.git'  # HTTPS — SSH keys are not available here
REPO_DIR = '/kaggle/working/rsna-knee'

import os, subprocess, sys
from pathlib import Path

if Path(REPO_DIR).exists():
    print(subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                         capture_output=True, text=True).stdout)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('HEAD:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                              capture_output=True, text=True).stdout.strip())

## 2. Locate the data

Found automatically rather than typed: `/kaggle/input` sometimes holds the competition
folder directly and sometimes nests it under `competitions/`, and pointing one level too
high is the easiest mistake to make.

In [ ]:
INPUT = Path('/kaggle/input')
candidates = sorted({p.parent for depth in (1, 2, 3)
                     for p in INPUT.glob('/'.join(['*'] * depth) + '/train.csv')})

print('candidates:', [str(c) for c in candidates] or 'NONE — is the dataset attached?')
assert len(candidates) == 1, 'pick one by hand below and set RSNA_RAW yourself'

os.environ['RSNA_RAW'] = str(candidates[0])
os.environ['RSNA_ARTIFACTS'] = '/kaggle/working/artifacts'

print('RSNA_RAW       =', os.environ['RSNA_RAW'])
print('RSNA_ARTIFACTS =', os.environ['RSNA_ARTIFACTS'])
print('contents       :', sorted(p.name for p in candidates[0].iterdir())[:10])

## 3. Dependencies

Installs only what is genuinely missing, so Kaggle's preinstalled numpy/torch build is left alone.

In [ ]:
!python scripts/colab_bootstrap.py --no-mount

In [ ]:
# Sanity check: the contracts (label order, fold rules, laterality) hold in this environment.
!python -m pytest -q

## 4. EDA (Task 1.2)

Reads one DICOM header per study to recover `PatientID` — this is what decides whether the
folds can group by patient. Roughly a minute per few thousand studies on Kaggle's local disk.

**Safe to re-run**: progress is checkpointed to `artifacts/eda/study_headers.parquet` every
500 studies, so an interrupted session resumes instead of restarting.

In [ ]:
!python scripts/run_eda.py

### The decision point

`PatientID` is not guaranteed to be among the 86 allowlisted tags. Read the number below
before freezing the folds:

* **≥ 95 %** → folds group by patient. Nothing to do.
* **< 95 %** → the tag is not published; grouping by study is then the only option and is
  fine (it means the organisers de-identified per study). Just know which one you froze.

In [ ]:
import pandas as pd

meta = pd.read_parquet('/kaggle/working/artifacts/eda/study_meta.parquet')
print('studies           :', len(meta))
print('PatientID coverage:', f"{meta['PatientID'].notna().mean():.1%}")
print('unique patients   :', meta['PatientID'].nunique(dropna=True))
print('site cluster src  :', meta['site_cluster_source'].value_counts().to_dict())

repeats = meta['PatientID'].value_counts()
print('patients with >1 study:', int((repeats > 1).sum()))

## 5. Frozen folds (Task 1.3)

Runs **once for the whole competition**. Re-running exits non-zero on purpose — check the
`group_key` line in the output.

In [ ]:
!python scripts/make_folds.py

In [ ]:
folds = pd.read_parquet('/kaggle/working/artifacts/folds.parquet')
print(folds['fold'].value_counts().sort_index().to_string())
print('\nlabeled studies:', int(folds['has_gt_labels'].sum()), '/', len(folds))
folds.head()

## 6. Read the report

Two tables matter for Spec 02: the **language census** (≥2 few-shot examples per major
language) and the **report length percentiles** (they set the LLM cost estimate). Then read
the risks section at the end.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(Path('docs/eda_report.md').read_text()))

## 7. Package the outputs

A few MB in total. Download the zip from the *Output* panel, then commit `folds.parquet`
and `docs/` to the repo — **`folds.parquet` is the one file that must never be regenerated**.

In [ ]:
import shutil

bundle = Path('/kaggle/working/spec01_outputs')
shutil.rmtree(bundle, ignore_errors=True)
(bundle / 'artifacts' / 'eda').mkdir(parents=True)

shutil.copy('/kaggle/working/artifacts/folds.parquet', bundle / 'artifacts')
for name in ('study_meta.parquet', 'study_headers.parquet'):
    shutil.copy(f'/kaggle/working/artifacts/eda/{name}', bundle / 'artifacts' / 'eda')
shutil.copytree('docs', bundle / 'docs')

archive = shutil.make_archive('/kaggle/working/spec01_outputs', 'zip', bundle)
print(archive, f'({Path(archive).stat().st_size / 1e6:.1f} MB)')

### If something failed

Every run writes a full log with a manifest (config hash, git SHA, seed, package versions).
The cell below shows the most recent one.

In [ ]:
logs = sorted(Path('/kaggle/working/artifacts/logs').glob('*.log'),
              key=lambda p: p.stat().st_mtime)
print(logs[-1] if logs else 'no logs yet')
print(logs[-1].read_text()[-4000:] if logs else '')